# シミュレーションとコンピュータグラフィックス

シミュレーションは「世界の中で何が起きるか」、CG は「それがこちらにどう見えるか」を担います。ここでいう CG は、動きの結果を画像や表示へ変える側の処理だと思ってください。状態遷移とは、今の状態から次の状態へどう進むかというルールのことです。世界モデルではこの 2 層を分けて考えることが実装上の要点です。


## 動きと見え方は、別の層で作られている

物体がどう動くかを決めるのはシミュレーションで、その結果がセンサーや画像としてどう見えるかを決めるのは描画側です。ここでいう観測とは、カメラ画像やセンサー値のように、外から実際に見えている情報のことです。最初は、観測を「こちらが受け取る見え方」だと思えば十分です。世界モデルを作るとき、この二つを混ぜると『動きの予想が悪いのか』『見せ方の変換が悪いのか』の切り分けが難しくなります。


ここでは放物運動のごく小さな例を使って、状態更新と 2D グリッドへの描画を分けて見ます。やりたいのは CG を学ぶことそのものではなく、『世界の動きの間違い』と『見え方の間違い』を別々に扱う感覚を持つことです。どこを直せばズレが減るのかを判断するための入口だと思ってください。最初は、『中で起きていること』と『こちらに見えていること』の二枚に分けて考えるだけで十分です。


## 反発係数ひとつでも、物理と観測は役割が違う

`vy *= -0.4` は床にぶつかったあとの跳ね返り方を決める物理側の設定です。一方で `canvas` に `*` を置く処理は、その状態をどんな観測へ変換するかという表示側の設定です。二つは見た目のズレとしては同じに見えても、片方は動きのルール、もう片方は見せ方のルールを直す話です。つまり、『どこで壊れたか』を見誤ると、直す場所もずれてしまいます。


## この notebook の見どころ

最初の状態列、グリッド上の軌跡、床反射の挙動を順に見ながら、どこまでが状態更新でどこからが見た目への変換かを切り分けます。読み終わったときに、『内部の軌跡は合っているか』『表示にした瞬間にずれたのか』を分けて言えれば十分です。


現実へ持っていくときに効くのも、この分離です。sim-to-real とは、シミュレーションで作った学習や計画を現実へ持っていくときに起きるズレの問題です。最初は「シミュレーションではうまくいったのに現実ではずれる問題」だと思えば十分です。『世界の動きが違うのか、見え方が違うのか』をまず切り分けないと調整方針が立ちません。


## 丸めが生む別種の誤差

連続座標をグリッドに落とすと、物理は合っていても見た目が粗くなります。連続座標とは、本来は小数まで含めて滑らかに動いている位置のことです。これは動きのルールそのものの失敗ではなく、観測へ変えるときに生じる粗さです。つまり、『中では滑らかに動いているのに、見える側では四捨五入されて荒く見える』という種類のズレです。学習器には別種のノイズとして入ります。ここで大事なのは、この粗さを見てすぐ物理モデルの失敗だと決めつけないことです。


## 読み方の軸

この notebook は物理エンジンや本格レンダラの代わりではありません。状態更新と観測生成を分けて設計する理由を、最小例で確かめるためのものです。世界の中で起きることと、それがこちらにどう見えるかを分けて考える練習だと思って進めてください。


## 状態を時間発展させる

まずは位置と速度を更新し続けて、世界の側で何が起きているかを数値として追います。ここではまだ絵にせず、『内部ではどう動いたことになっているか』だけを見ます。予測の本体はまずこちらにあります。もしここですでに変なら、後ろの表示側を直しても根本解決にはなりません。


In [ ]:
import numpy as np

dt = 0.1
g = -9.8
x, y = 0.0, 0.0
vx, vy = 2.0, 6.0
traj = []
for _ in range(25):
    x += vx * dt
    y += vy * dt
    vy += g * dt
    if y < 0:
        y = 0
        vy *= -0.4
    traj.append((x, y))
print('first 5 simulated states =', [(round(a,2), round(b,2)) for a,b in traj[:5]])


## 数値の軌跡を見える形にする

次に、その状態列を粗いキャンバスへ写して、観測として何が見えているかを確かめます。同じ動きでも、どんな解像度でどう描くかによって見た目は変わります。ここで初めて『外からどう見えるか』が加わります。内部の軌跡が同じでも、観測側の作り方しだいで学習器が受け取る情報はかなり変わります。


In [ ]:
# CG: 連続状態をグリッドに投影して可視化
W, H = 40, 12
canvas = [['.' for _ in range(W)] for _ in range(H)]

for x, y in traj:
    px = min(W - 1, max(0, int(x * 3)))
    py = min(H - 1, max(0, int(y * 1.5)))
    canvas[H - 1 - py][px] = '*'

for row in canvas:
    print(''.join(row))


この分離を保っておくと、ズレが出たときに『動きのルールを直すのか、見せ方を直すのか』を判断しやすくなります。要するに、同じ見た目の失敗でも、世界の中のルールを直すべきか、こちらへの見せ方を直すべきかは別問題です。世界モデルの実装では、この切り分け自体がかなり重要です。デバッグでも、まず内部状態を見てから観測表示を見る順にすると原因を追いやすくなります。
